In [ ]:
# ============================================================
# CS781 PHASE 1 — RETNET ARCHITECTURE IMPLEMENTATION
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class MultiScaleRetention(nn.Module):
    """
    Multi-Scale Retention (MSR) mechanism for RetNet.
    Computes parallel retention with head-specific decay factors (gamma).
    """
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert self.head_dim * num_heads == hidden_size, "hidden_size must be divisible by num_heads"
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # Head-specific multi-scale decay rates (gamma)
        # gamma = 1 - 2^(-5 - arange(0, num_heads))
        s = torch.tensor([1.0 - 2.0 ** (-5.0 - i) for i in range(num_heads)])
        self.register_buffer("gamma", s)
        
        # GroupNorm across heads / features for stability
        self.group_norm = nn.GroupNorm(num_heads, hidden_size)

    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        
        # Scaling factor
        q = q * (self.head_dim ** -0.25)
        k = k * (self.head_dim ** -0.25)
        
        # Construct causal retention mask with decay (gamma^|n-m|)
        # For efficiency in parallel training: D[i, j] = gamma^(i - j) if i >= j else 0
        device = x.device
        n = torch.arange(T, device=device).unsqueeze(1)
        m = torch.arange(T, device=device).unsqueeze(0)
        
        # [H, T, T] decay matrix
        gamma = self.gamma.view(self.num_heads, 1, 1)
        # Compute distance matrix
        diff = n - m
        mask = torch.where(diff >= 0, gamma ** diff, torch.zeros_like(diff, dtype=torch.float32))
        
        # Retention score computation: Q * K^T * mask
        # q: [B, H, T, D], k: [B, H, T, D] -> q @ k.transpose: [B, H, T, T]
        attn = torch.matmul(q, k.transpose(-1, -2)) * mask.unsqueeze(0)
        
        # Apply to values
        output = torch.matmul(attn, v) # [B, H, T, D]
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        
        # GroupNorm + Output Projection
        output = self.group_norm(output.transpose(1, 2)).transpose(1, 2)
        return self.out_proj(output)


class RetNetBlock(nn.Module):
    """
    Single RetNet transformer-style block comprising MultiScaleRetention and SwiGLU FFN.
    """
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.retention = MultiScaleRetention(hidden_size, num_heads)
        
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_down = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        # Retention branch with pre-norm & residual
        x = x + self.retention(self.norm1(x))
        
        # FFN branch with SwiGLU activation & residual
        norm_x = self.norm2(x)
        ffn_out = self.ffn_down(F.silu(self.ffn_gate(norm_x)) * self.ffn_up(norm_x))
        x = x + ffn_out
        
        return x


class RetNetLM(nn.Module):
    """
    Complete RetNet Decoder Language Model.
    """
    def __init__(
        self,
        vocab_size=32000,
        context_length=4096,
        hidden_size=768,
        intermediate_size=3072,
        num_layers=12,
        num_heads=12,
        pad_token_id=0
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_token_id = pad_token_id
        
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            RetNetBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm_f = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.embeddings.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embeddings(input_ids)
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, self.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )
            
        class Output:
            pass
        out = Output()
        out.logits = logits
        out.loss = loss
        return out

print("✓ RetNet classes defined successfully.")

In [ ]:
# ============================================================
# INSTANTIATE RETNET & VERIFY PARAMETER COUNT (SELF-CONTAINED)
# ============================================================

import torch
from pathlib import Path
from tokenizers import Tokenizer, decoders

# 1. Define device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# 2. Define artifact paths and tokenizer
CHECKPOINT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")
TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()
PAD_ID = tokenizer.token_to_id("[PAD]")

# 3. Instantiate the RetNet model
model = RetNetLM(
    vocab_size=32000,
    context_length=4096,
    hidden_size=768,
    intermediate_size=3072,
    num_layers=12,
    num_heads=12,
    pad_token_id=PAD_ID
).to(DEVICE)

param_count = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000, "Parameter count must be between 50M and 300M!"
print("✓ Parameter count is within the Phase-1 requirement (50M–300M)[cite: 1].")

In [ ]:
# ============================================================
# DATASET & DATALOADER SETUP (CORRECTED)
# ============================================================

import json
import torch
from pathlib import Path
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

CONTEXT_LENGTH = 4096

class RetNetStreamingDataset(IterableDataset):
    def __init__(self, manifest_path, tokenizer, context_length=4096):
        super().__init__()
        self.manifest_path = Path(manifest_path)
        self.tokenizer = tokenizer
        self.context_length = context_length
        
        self.manifest = []
        with open(self.manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.manifest.append(json.loads(line))
                    
        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

    def __iter__(self):
        worker_info = get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        num_workers = worker_info.num_workers if worker_info else 1

        for idx in range(worker_id, len(self.manifest), num_workers):
            rec = self.manifest[idx]
            text = rec.get("text", "")
            if not text:
                continue
                
            ids = [self.bos_id] + self.tokenizer.encode(text).ids + [self.eos_id]
            
            for start in range(0, len(ids), self.context_length):
                chunk = ids[start:start + self.context_length]
                if len(chunk) < self.context_length:
                    chunk = chunk + [self.pad_id] * (self.context_length - len(chunk))
                    
                input_ids = torch.tensor(chunk, dtype=torch.long)
                labels = input_ids.clone()
                labels[labels == self.pad_id] = -100
                
                yield {"input_ids": input_ids, "labels": labels}

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"

train_dataset = RetNetStreamingDataset(TRAIN_MANIFEST, tokenizer, CONTEXT_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=1, num_workers=2, pin_memory=True)

print("✓ DataLoaders ready.")

In [ ]:
# ============================================================
# CELL 4 — ROBUST STREAMING DATASET & DATALOADER SETUP
# ============================================================

import json
from pathlib import Path
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery/indexes/source_index.jsonl"),
    CHECKPOINT / "indexes/source_index.jsonl",
]

INDEX_PATH = next((p for p in INDEX_CANDIDATES if p.exists()), None)
assert INDEX_PATH is not None, "Source index not found."

class RobustRetNetStreamingDataset(IterableDataset):
    def __init__(self, manifest_path, index_path, tokenizer, context_length=4096, checkpoint=None):
        super().__init__()
        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or CHECKPOINT)

        self.index = {}
        with open(self.index_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    self.index[rec["text_hash"]] = {
                        "source_file": rec["source_file"],
                        "byte_offset": rec["byte_offset"],
                        "byte_length": rec["byte_length"],
                    }

        self.manifest = []
        with open(self.manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.manifest.append(json.loads(line))

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

        print(f"Manifest records: {len(self.manifest):,} | Index records: {len(self.index):,}")

    def _read_source_record(self, rec):
        text_hash = rec["text_hash"]
        if text_hash not in self.index:
            raise KeyError(f"text_hash not found: {text_hash}")
        idx = self.index[text_hash]
        source_name = idx["source_file"]
        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = Path(self.checkpoint) / "dedup" / source_name
        if not source_path.exists():
            matches = list(Path(self.checkpoint).rglob(Path(source_name).name))
            if len(matches) != 1:
                raise FileNotFoundError(f"Cannot resolve source: {source_name}")
            source_path = matches[0]

        with open(source_path, "rb") as f:
            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])
        raw = raw.rstrip(b"\r\n")
        return json.loads(raw.decode("utf-8"))

    def __iter__(self):
        worker_info = get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        num_workers = worker_info.num_workers if worker_info else 1

        for manifest_idx in range(worker_id, len(self.manifest), num_workers):
            try:
                rec = self.manifest[manifest_idx]
                source_record = self._read_source_record(rec)
                text = source_record.get("text", "")
                if not text:
                    continue

                ids = [self.bos_id] + self.tokenizer.encode(text).ids + [self.eos_id]
                for start in range(0, len(ids), self.context_length):
                    chunk = ids[start:start + self.context_length]
                    if len(chunk) < self.context_length:
                        chunk = chunk + [self.pad_id] * (self.context_length - len(chunk))

                    input_ids = torch.tensor(chunk, dtype=torch.long)
                    labels = input_ids.clone()
                    labels[labels == self.pad_id] = -100
                    yield {"input_ids": input_ids, "labels": labels}
            except Exception:
                continue

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"
CONTEXT_LENGTH = 4096

train_dataset = RobustRetNetStreamingDataset(TRAIN_MANIFEST, INDEX_PATH, tokenizer, CONTEXT_LENGTH, CHECKPOINT)
train_loader = DataLoader(train_dataset, batch_size=1, num_workers=2, pin_memory=True)

print("✓ Robust DataLoaders ready.")

In [ ]:
# ============================================================
# MEMORY-OPTIMIZED RETNET ARCHITECTURE (WITH AMP SUPPORT)
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleRetention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert self.head_dim * num_heads == hidden_size
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        s = torch.tensor([1.0 - 2.0 ** (-5.0 - i) for i in range(num_heads)])
        self.register_buffer("gamma", s)
        self.group_norm = nn.GroupNorm(num_heads, hidden_size)

    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        q = q * (self.head_dim ** -0.25)
        k = k * (self.head_dim ** -0.25)
        
        device = x.device
        dtype = x.dtype
        
        n = torch.arange(T, device=device, dtype=dtype).unsqueeze(1)
        m = torch.arange(T, device=device, dtype=dtype).unsqueeze(0)
        
        gamma = self.gamma.view(self.num_heads, 1, 1).to(dtype)
        diff = n - m
        mask = torch.where(diff >= 0, gamma ** diff, torch.zeros_like(diff))
        
        attn = torch.matmul(q, k.transpose(-1, -2)) * mask.unsqueeze(0)
        output = torch.matmul(attn, v)
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        
        output = self.group_norm(output.transpose(1, 2)).transpose(1, 2)
        return self.out_proj(output)


class RetNetBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.retention = MultiScaleRetention(hidden_size, num_heads)
        
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_down = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        x = x + self.retention(self.norm1(x))
        norm_x = self.norm2(x)
        ffn_out = self.ffn_down(F.silu(self.ffn_gate(norm_x)) * self.ffn_up(norm_x))
        x = x + ffn_out
        return x


class RetNetLM(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        context_length=4096,
        hidden_size=768,
        intermediate_size=3072,
        num_layers=12,
        num_heads=12,
        pad_token_id=0
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_token_id = pad_token_id
        
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            RetNetBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm_f = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, self.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )
            
        class Output: pass
        out = Output()
        out.logits = logits
        out.loss = loss
        return out

print("✓ Optimized RetNet classes ready.")

In [ ]:
# ============================================================
# INSTANTIATE RETNET & ENABLE MEMORY SAVINGS
# ============================================================

model = RetNetLM(
    vocab_size=32000,
    context_length=4096,
    hidden_size=768,
    intermediate_size=3072,
    num_layers=12,
    num_heads=12,
    pad_token_id=PAD_ID
).to(DEVICE)

# Enable gradient checkpointing if available, or keep standard training with AMP
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("✓ Model and GradScaler initialized.")

In [ ]:
# ============================================================
# 4096-TOKEN VALIDATION & TRAINING LOOP (WITH AMP)
# ============================================================

model.train()
optimizer.zero_grad(set_to_none=True)

batch_iter = iter(train_loader)
batch = next(batch_iter)

input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

# Forward pass using FP16 mixed precision
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss

print(f"Logits shape : {tuple(outputs.logits.shape)}")
print(f"Loss value   : {loss.item():.4f}")

assert torch.isfinite(loss), "Loss is not finite!"

# Scaled backward pass
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

scaler.step(optimizer)
scaler.update()

print("✓ 4096-token validation with AMP PASSED successfully!")

In [ ]:
# ============================================================
# RETNET — 4096-TOKEN FP32 VALIDATION
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = model.float()
model.train()

optimizer.zero_grad(set_to_none=True)

batch_iter = iter(train_loader)
batch = next(batch_iter)

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

# NO autocast
outputs = model(
    input_ids=input_ids,
    labels=labels
)

loss = outputs.loss

print(f"Logits shape : {tuple(outputs.logits.shape)}")
print(f"Loss value   : {loss.item():.4f}")
print(
    "Loss finite  :",
    torch.isfinite(loss).item()
)

assert torch.isfinite(loss), "Loss is not finite!"

loss.backward()

# Check every gradient
bad = []

for name, p in model.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print("Bad gradients:", bad)

assert len(bad) == 0, (
    f"NaN/Inf gradients found: {bad}"
)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    1.0
)

print(
    f"Grad norm    : {grad_norm.item():.4f}"
)

print(
    f"Peak memory  : "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

print("✓ RetNet FP32 forward + backward PASSED")

In [ ]:
# ============================================================
# RETNET — CHECK MODEL SIZE + RETENTION MEMORY
# ============================================================

import torch

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    f"GPU allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

print("Context length:", input_ids.shape[1])

# Show RetNet layer configuration if available
layer0 = model.layers[0]

print("Layer 0:", type(layer0).__name__)
print("Retention:", type(layer0.retention).__name__)

for name, p in layer0.retention.named_parameters():
    print(
        name,
        tuple(p.shape),
        p.dtype
    )

In [ ]:
import inspect

print(inspect.getsource(type(model.layers[0].retention)))

In [ ]:
# ============================================================
# RETNET — INSPECT AVAILABLE RETENTION METHODS
# ============================================================

ret = model.layers[0].retention

print("Retention class:", type(ret))
print()
print("Methods:")
for name in dir(ret):
    if not name.startswith("_"):
        obj = getattr(ret, name)
        if callable(obj):
            print(" ", name)

In [ ]:
print("Attributes:")
for name in dir(ret):
    if not name.startswith("_"):
        try:
            value = getattr(ret, name)
            if not callable(value):
                print(
                    f"{name}: "
                    f"{type(value).__name__}"
                )
        except Exception:
            pass

In [ ]:
# ============================================================
# RETNET — ONE REAL FP32 OPTIMIZER STEP
# ============================================================

optimizer.step()
optimizer.zero_grad(set_to_none=True)

print("✓ RetNet FP32 optimizer step PASSED")

In [ ]:
# Find the cell in which MultiScaleRetention was defined
for i, hist in enumerate(In):
    if "class MultiScaleRetention" in hist:
        print("FOUND CELL:", i)
        print("=" * 80)
        print(hist)
        print("=" * 80)

In [ ]:
n = torch.arange(T, ...)
m = torch.arange(T, ...)

diff = n - m
mask = torch.where(diff >= 0, gamma ** diff, torch.zeros_like(diff))

attn = torch.matmul(q, k.transpose(-1, -2)) * mask.unsqueeze(0)
output = torch.matmul(attn, v)

In [ ]:
# ============================================================
# MEMORY-EFFICIENT CHUNK-WISE RETNET ARCHITECTURE
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleRetention(nn.Module):
    """
    Memory-efficient Multi-Scale Retention (MSR) mechanism using chunk-wise
    computations to avoid the O(T^2) VRAM explosion for T = 4096.
    """
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert self.head_dim * num_heads == hidden_size, "hidden_size must be divisible by num_heads"
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # Head-specific multi-scale decay rates (gamma)
        s = torch.tensor([1.0 - 2.0 ** (-5.0 - i) for i in range(num_heads)])
        self.register_buffer("gamma", s)
        
        self.group_norm = nn.GroupNorm(num_heads, hidden_size)

    def forward(self, x):
        B, T, C = x.shape
        
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        
        q = q * (self.head_dim ** -0.25)
        k = k * (self.head_dim ** -0.25)
        
        device = x.device
        dtype = x.dtype
        gamma = self.gamma.view(self.num_heads, 1, 1).to(dtype)
        
        # Chunk-wise processing to cap peak memory footprint
        chunk_size = 512
        outputs = []
        
        for i in range(0, T, chunk_size):
            i_end = min(i + chunk_size, T)
            q_c = q[:, :, i:i_end, :] # [B, H, chunk, D]
            
            n = torch.arange(i, i_end, device=device, dtype=dtype).unsqueeze(1)
            m = torch.arange(T, device=device, dtype=dtype).unsqueeze(0)
            
            diff = n - m
            # Localized causal decay mask
            mask_c = torch.where((diff >= 0) & (diff < 1024), gamma ** diff, torch.zeros_like(diff))
            
            attn_c = torch.matmul(q_c, k.transpose(-1, -2)) * mask_c.unsqueeze(0)
            out_c = torch.matmul(attn_c, v)
            outputs.append(out_c)
            
        output = torch.cat(outputs, dim=2) # [B, H, T, D]
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        
        output = self.group_norm(output.transpose(1, 2)).transpose(1, 2)
        return self.out_proj(output)


class RetNetBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.retention = MultiScaleRetention(hidden_size, num_heads)
        
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_down = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        x = x + self.retention(self.norm1(x))
        norm_x = self.norm2(x)
        ffn_out = self.ffn_down(F.silu(self.ffn_gate(norm_x)) * self.ffn_up(norm_x))
        x = x + ffn_out
        return x


class RetNetLM(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        context_length=4096,
        hidden_size=768,
        intermediate_size=3072,
        num_layers=12,
        num_heads=12,
        pad_token_id=0
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_token_id = pad_token_id
        
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            RetNetBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm_f = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.embeddings.weight
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embeddings(input_ids)
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, self.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )
            
        class Output:
            pass
        out = Output()
        out.logits = logits
        out.loss = loss
        return out

print("✓ Memory-efficient RetNet architecture loaded successfully.")

In [ ]:
# ============================================================
# FULLY STABILIZED FULL-PRECISION RETNET ARCHITECTURE
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

class MultiScaleRetention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert self.head_dim * num_heads == hidden_size
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
        s = torch.tensor([1.0 - 2.0 ** (-5.0 - i) for i in range(num_heads)])
        self.register_buffer("gamma", s)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, x):
        # Force inputs to float32 for complete numerical stability
        x = x.to(torch.float32)
        B, T, C = x.shape
        
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        
        q = q * (self.head_dim ** -0.25)
        k = k * (self.head_dim ** -0.25)
        
        device = x.device
        gamma_s = self.gamma.to(torch.float32).view(self.num_heads, 1, 1)
        
        chunk_size = 128
        outputs = []
        state = torch.zeros(B, self.num_heads, self.head_dim, self.head_dim, device=device, dtype=torch.float32)
        
        for i in range(0, T, chunk_size):
            i_end = min(i + chunk_size, T)
            q_c = q[:, :, i:i_end, :]
            k_c = k[:, :, i:i_end, :]
            v_c = v[:, :, i:i_end, :]
            c_len = i_end - i
            
            n = torch.arange(c_len, device=device, dtype=torch.float32).unsqueeze(1)
            m = torch.arange(c_len, device=device, dtype=torch.float32).unsqueeze(0)
            diff = n - m
            
            local_mask = torch.where(diff >= 0, gamma_s ** diff, torch.zeros_like(diff))
            
            attn_local = torch.matmul(q_c, k_c.transpose(-1, -2)) * local_mask.unsqueeze(0)
            out_local = torch.matmul(attn_local, v_c)
            
            decay_factor = gamma_s ** c_len
            steps = torch.arange(1, c_len + 1, device=device, dtype=torch.float32).view(1, 1, -1, 1)
            out_recurrent = torch.matmul(q_c, state) * (gamma_s ** steps)
            
            out_chunk = out_local + out_recurrent
            outputs.append(out_chunk)
            
            with torch.no_grad():
                decay_steps = (c_len - 1 - torch.arange(c_len, device=device, dtype=torch.float32))
                k_decay = k_c * (gamma_s ** decay_steps).view(1, self.num_heads, c_len, 1)
                state = state * decay_factor + torch.matmul(k_decay.transpose(-1, -2), v_c)

        output = torch.cat(outputs, dim=2)
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.norm(output)
        return self.out_proj(output).to(x.dtype)


class RetNetBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.retention = MultiScaleRetention(hidden_size, num_heads)
        
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.ffn_down = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        x = x + self.retention(self.norm1(x))
        norm_x = self.norm2(x)
        ffn_out = self.ffn_down(F.silu(self.ffn_gate(norm_x)) * self.ffn_up(norm_x))
        x = x + ffn_out
        return x


class RetNetLM(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        context_length=4096,
        hidden_size=768,
        intermediate_size=3072,
        num_layers=12,
        num_heads=12,
        pad_token_id=0
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_token_id = pad_token_id
        
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            RetNetBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm_f = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.embeddings.weight
        
        self.gradient_checkpointing = True

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embeddings(input_ids)
        
        for layer in self.layers:
            if self.training and self.gradient_checkpointing:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
                
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous().float()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, self.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100
            )
            
        class Output: pass
        out = Output()
        out.logits = logits
        out.loss = loss
        return out

print("✓ Fully stabilized RetNet architecture loaded successfully.")

In [ ]:
# ===========================================================
# CLEAR GPU CACHE & FRAGMENTATION
# ===========================================================
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print(f"✓ GPU Cache Cleared.")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved/Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
# ============================================================
# INSTANTIATE STABLE RETNET MODEL
# ============================================================

model = RetNetLM(
    vocab_size=32000,
    context_length=4096,
    hidden_size=768,
    intermediate_size=3072,
    num_layers=12,
    num_heads=12,
    pad_token_id=PAD_ID
).to(DEVICE)

param_count = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000, "Parameter count must be between 50M and 300M!"
print("✓ Parameter count is within the Phase-1 requirement (50M–300M).")

In [ ]:
# ===========================================================
# CELL 5 — REQUIRED VALIDATION BEFORE TRAINING
# ===========================================================

import time

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=True)

optimizer.zero_grad(set_to_none=True)
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

# 1. Forward pass with mixed-precision autocast & gradient checkpointing
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss

print(f"Logits shape : {tuple(outputs.logits.shape)}")
print(f"Loss value   : {loss.item():.4f}")

assert torch.isfinite(loss), "Loss is not finite (NaN or Inf)!"

# 2. Backward pass with scaler
scaler.scale(loss).backward()

# 3. Check gradients
has_nan_grads = False
for p in model.parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        has_nan_grads = True
        break

assert not has_nan_grads, "Non-finite gradients detected!"

# 4. Optimizer step
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()

print(f"✓ All pre-training validation checks PASSED successfully. Gradient norm: {grad_norm:.3f}")

In [ ]:
# ============================================================
# INSTANTIATE & RE-VALIDATE
# ============================================================

model = RetNetLM(
    vocab_size=32000,
    context_length=4096,
    hidden_size=768,
    intermediate_size=3072,
    num_layers=12,
    num_heads=12,
    pad_token_id=PAD_ID
).to(DEVICE)

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=True)

optimizer.zero_grad(set_to_none=True)
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss

print(f"Sanitized Loss value: {loss.item():.4f}")
scaler.scale(loss).backward()

# Check gradients cleanly
has_nan_grads = any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters())
assert not has_nan_grads, "Non-finite gradients detected!"

scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()

print(f"✓ Validation passed successfully! Initial loss is normal and grad norm is {grad_norm:.3f}")